# File Formats & ETL Overview

## Overview
This document provides a concise explanation of common data file formats (CSV and JSON) and the ETL (Extract → Transform → Load) process. It covers characteristics, advantages, limitations, and examples to help understand how data is stored, processed, and moved across systems.

---

# 1. File Formats

## 1.1 CSV (Comma-Separated Values)

### **Overview**
CSV is a lightweight, text-based format used to store tabular data.  
Each row represents a record, and each column is separated by a comma.

### **Example**
```csv
name,age,city
Aakash,21,Kathmandu
Bikash,22,Pokhara
```

### **Characteristics**
- Plain text format  
- Represents data in rows and columns  
- Does not support explicit data types  
- No nested structures

### **Advantages**
- Easy to read and parse  
- Supported by almost all tools  
- Lightweight and fast  

### **Limitations**
- Cannot store nested objects  
- No support for metadata  
- Data type information is lost  

---

## 1.2 JSON (JavaScript Object Notation)

### **Overview**
JSON is a flexible structured format widely used in APIs, configuration files, and modern applications.

### **Example**
```json
{
  "name": "Aakash",
  "age": 21,
  "address": {
    "city": "Kathmandu",
    "country": "Nepal"
  },
  "skills": ["Python", "Django", "ML"]
}
```

### **Characteristics**
- Key–value representation  
- Supports nested objects and arrays  
- Human- and machine-readable  
- Language-independent

### **Advantages**
- Represents complex and hierarchical data  
- Self-descriptive  
- Standard format for APIs  

### **Limitations**
- More verbose than CSV  
- Slightly heavier to parse  
- Not ideal for purely tabular data  

---

# 2. ETL (Extract → Transform → Load)

## 2.1 Extract

### **Overview**
Extraction is the process of collecting raw data from multiple sources.

### **Common Sources**
- Databases (SQL / NoSQL)  
- CSV / JSON / XML files  
- APIs  
- Web scraping  
- Cloud storage (S3, GCS, BigQuery)

### **Goal**
Collect all the raw data required for processing.

---

## 2.2 Transform

### **Overview**
Transformation converts raw data into clean, usable, and structured formats.

### **Common Transformations**
- Data cleaning (remove duplicates, handle missing values)  
- Data type conversions  
- Scaling / normalization  
- Merging datasets  
- Filtering / aggregation  
- Feature engineering (for machine learning)

### **Goal**
Prepare structured, high-quality data ready for storage, analysis, or modeling.

---

## 2.3 Load

### **Overview**
Loading moves transformed data into a target system for storage or analysis.

### **Common Targets**
- Databases (PostgreSQL, MySQL)  
- Data warehouses (Snowflake, Redshift, BigQuery)  
- Analytics tools (Power BI, Tableau)  
- Machine learning pipelines  

### **Goal**
Store processed data for reporting, analytics, or machine learning workflows.


**1. Write a save_to_csv(data, filename) function that saves your cleaned list of dictionaries into a CSV file.**

In [1]:
import csv
import json
import re

# ============================================================
# 1. Saving Functions
# ============================================================

def save_to_csv(data, filename):
    """Save list of dictionaries to a CSV file."""
    if not data:
        print("No data to save.")
        return
    
    headers = data[0].keys()

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(data)

    print(f"CSV saved: {filename}")

**2. Write a save_to_json(data, filename) function that saves the same cleaned data into a JSON file.**
     

In [2]:
import csv
import json
import re

def save_to_json(data, filename):
    """Save list of dictionaries to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    print(f"JSON saved: {filename}")


**3. Create an extract() function that loads raw data (can be a list, file, or dummy input for testing).**

In [3]:
import csv
import json
import re

def extract():
    """Simulate raw data input."""
    raw_items = [
        {"title": "Amazing Product!!! 😍", "price": "₹1,299", "rating": "Rated 4.5 out of 5"},
        {"title": "Cheap & Good$ Item", "price": "$499", "rating": "3/5"},
        {"title": "Luxury Item $$$", "price": "£2,000", "rating": "Rated: 5"},
        {"title": "@@@Invalid###", "price": "", "rating": "None"}  # dirty record
    ]

    return raw_items


**4. Create a transform() function that uses your Day-3 cleaning functions to clean all records.**

In [4]:
import csv
import json
import re

def clean_price(price):
    """Remove symbols like ₹, $, £ and convert to float."""
    price_numeric = re.sub(r'[₹$£,]', '', price).strip()
    return float(price_numeric) if price_numeric else 0.0


def clean_rating(text):
    """Extract numeric rating such as 4.5 from text."""
    match = re.search(r'\d+(\.\d+)?', text)
    return float(match.group()) if match else 0.0


def clean_text(text):
    """Remove emojis, symbols, keep only alphanumeric."""
    clean = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return re.sub(r'\s+', ' ', clean).strip()


def clean_item_dict(item):
    """Clean a single dictionary (one product)."""
    return {
        'title': clean_text(item.get('title', '')),
        'price': clean_price(item.get('price', '0')),
        'rating': clean_rating(item.get('rating', '0'))
    }



def transform(raw_data):
    cleaned_data = []
    for item in raw_data:
        cleaned_data.append(clean_item_dict(item))
    return cleaned_data



**5. Create a load() function that takes cleaned data and writes it both to CSV and JSON.**

In [5]:
import csv
import json
import re

def load(cleaned_data):
    save_to_csv(cleaned_data, "cleaned_output.csv")
    save_to_json(cleaned_data, "cleaned_output.json")

**6 and 7. Build a simple etl_pipeline():
• call extract()
• pass results to transform()
• pass cleaned data to load()
• return final output
Print:
• number of records extracted
• number of records cleaned
• confirmation that CSV and JSON were created successfully**

In [6]:
import csv
import json
import re

def etl_pipeline():
    # Extract
    raw = extract()
    print("Extracted records:", len(raw))

    # Transform
    cleaned = transform(raw)
    print("Cleaned records:", len(cleaned))

    # Load
    load(cleaned)

    print("\nETL completed successfully!")
    return cleaned


# Run the pipeline

if __name__ == "__main__":
    final_output = etl_pipeline()
    print("\nFinal Cleaned Output:")
    print(final_output)


Extracted records: 4
Cleaned records: 4
CSV saved: cleaned_output.csv
JSON saved: cleaned_output.json

ETL completed successfully!

Final Cleaned Output:
[{'title': 'Amazing Product', 'price': 1299.0, 'rating': 4.5}, {'title': 'Cheap Good Item', 'price': 499.0, 'rating': 3.0}, {'title': 'Luxury Item', 'price': 2000.0, 'rating': 5.0}, {'title': 'Invalid', 'price': 0.0, 'rating': 0.0}]
